[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BradSegal/OxfordCDT2026ClinicalUtility/blob/main/solutions/01_clinical_utility.ipynb)

In [ ]:
# Setup: run this cell first. It installs the course helpers (about 20 seconds).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("clinical_utility") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/BradSegal/OxfordCDT2026ClinicalUtility.git"], check=True)
    importlib.invalidate_caches()

import plotly.io as pio

try:
    from google.colab import output
    output.enable_custom_widget_manager()
    pio.renderers.default = "colab"
except ImportError:
    pio.renderers.default = "plotly_mimetype"
print("Course helpers ready.")

# From AF scores to clinical decisions

You have trained ECG classifiers that output a score for each recording. A
service cannot act on a score. It has to decide, recording by recording,
whether to **send the recording for AF review**. This practical is about that
decision: what each outcome costs, how a threshold follows from those prices,
and why the best threshold or model changes when the prices change.

By the end you should be able to:

- derive a decision threshold from a priced decision matrix;
- read sensitivity, specificity, PPV and NPV through their denominators;
- price a policy and explain why maximising Youden's J and minimising expected
  loss choose different cutoffs;
- show that the preferred model can depend on the prices, whatever the AUC;
- derive net benefit from the same matrix and explain why calibration matters
  at a stated threshold probability.

**About the data.** Recording IDs and reference labels come from
[PhysioNet/CinC 2017](https://physionet.org/content/challenge-2017/1.0.0/)
and are real. Every model score is simulated for teaching. "Random forest" and
"ResNet" name plausible approaches; no model was trained and nothing here
measures how such models perform. Teaching A to F are invented recordings.
Prices are illustrative loss units, not health economics.

## How to work

Work in pairs. Agree a prediction before either of you types, and swap who
types at each section.

Each problem is labelled **Predict**, **Make** or **Fix the bug**. The checker
under it prints ✓, a "Not yet" hint naming the likely mistake, or "Not
attempted yet" while the `...` remains. Problems 7 and 13 ask for a
hypothesis before you have the evidence: their checkers record your answer
without marking it, and the next cell shows the result beside it. Checkers
never stop the notebook. Folded explanations are best opened after your own
attempt. Every widget has an ordinary function call in the next cell that
does the same job.

**Stuck for about 10 minutes?** Move on. Below each function that later cells
need is a commented line such as
`# decision_cost = check.use_reference("decision_cost")`. Delete the `#` and
run it to carry on with the reference version. A later cell that needs an
unfinished function prints `Waiting for ...` and names the problem. Once that
problem is done or switched, click in the waiting cell, choose **Runtime > Run
before**, then run the waiting cell: earlier cells that waited for the same
problem run again first. Worked copies of every notebook are in the
repository's `solutions/` folder.

| Section | Time | Problems |
|---|---|---|
| 1. One recording, two actions | about 10 minutes | 1 |
| 2. Six teaching recordings | about 10 minutes | 2 and 3 |
| 3. Price one decision | about 5 minutes | 4 |
| 4. A policy and two objectives | about 15 minutes | 5 to 7 |
| 5. Which model? It depends on the prices | about 10 minutes | 8 and 9 |
| 6. Fix the bug | about 5 minutes | 10 and 11 |
| 7. From prices to decision curves | about 10 minutes | 12 and 13 |
| 8. Freeze and defend a policy | about 5 minutes | 14 |
| 9. Closing discussion | about 10 minutes | |

The times are rough guides. **If you are behind, skip these** and come back
to them later: Problem 3 (read its folded answer instead), the six-recording
explorer in Section 3, Problem 9 and the resampling cell after it, and the
decision-curve explorer in Section 7 (its plain figure is enough). Do not skip
Section 8 or the closing discussion.

### Notation

| Term | Meaning |
|---|---|
| review | Send the recording for AF review: a clinician reads the ECG. |
| `TP`, `FP`, `FN`, `TN` | Counts of recordings with each outcome: reviewed with AF, reviewed without AF, AF not reviewed (a miss), and neither. |
| $C_{TP}$, $C_{FP}$, $C_{FN}$, $C_{TN}$ | The price of one recording with that outcome. In code, `costs["FN"]` holds $C_{FN}$. |
| loss | The sum of the prices over a set of decisions. Lower is better. |
| score | A model's output for one recording. It is a probability only if it is calibrated. |
| cutoff | Review when score ≥ cutoff. Functions call it `threshold`. |
| $p$ | One recording's probability of AF. |
| $p_t$ | The threshold probability: the AF probability at which both actions have the same expected loss. Decision curves call it the **preference**. |
| $n_{pos}$, $n_{neg}$, $n$ | AF recordings, recordings without AF, and all recordings. |
| random forest, raw ResNet, recalibrated ResNet | The three models. The recalibrated ResNet is the raw ResNet's scores passed through a fitted map (Section 7). |

In [ ]:
from IPython.display import display
from clinical_utility import data, reference, comparison, plots, checks as check

costs = {"TP": 1, "FP": 1, "FN": 5, "TN": 0}
records = data.teaching_cases()

## 1. One recording, two actions (about 10 minutes)

The action is to send a recording for AF review: a clinician reads the ECG.
A positive call is not a diagnosis or a treatment decision. A negative call
means the recording does not enter the review pathway. It does not establish
that the person has no AF.

Crossing the action with the reference label gives four outcomes, each with a
price.

| | Reference AF | No reference AF |
|---|---:|---:|
| **Review** | $C_{TP}$ = 1 | $C_{FP}$ = 1 |
| **No review** | $C_{FN}$ = 5 | $C_{TN}$ = 0 |

A review takes the same clinician time whether or not it finds AF, so
$C_{TP}$ and $C_{FP}$ are both 1. $C_{FP}$ is the whole price of a review that
finds no AF; here we assume no further harm. A miss is priced at 5, so
reviewing an AF recording saves 4: the miss is avoided, but the review still
costs 1. The service makes this valuation; the ECG does not supply it. The
`costs` dictionary above holds the matrix.

### Before the outcome is known

Suppose a recording has AF probability $p$. The expected loss of an action
averages its two possible outcomes:

$$E[L(\text{review})] = 1\cdot p + 1\cdot(1-p) = 1, \qquad
E[L(\text{no review})] = 5p + 0\cdot(1-p) = 5p.$$

At an AF probability of 10%, review costs 1 and no review costs 0.5, so no
review is preferred. At 60%, they cost 1 and 3, so review is preferred. The
two are equal when $1 = 5p$, that is at $p = 0.2$. This is the threshold
probability $p_t$. We review at equality, where either action has the same
expected loss.

This argument needs a probability that means what it says. A score of 0.2 is
not automatically a 20% chance of AF. Section 7 returns to that.

In [ ]:
display(plots.expected_cost_figure(costs))

### Problem 1: Predict

Suppose $C_{FN}$ = 10 instead of 5, and the other three prices are unchanged.
Before you touch the explorer, work out the new threshold probability $p_t$
and store it as a proportion. Then tell your partner which way it moved, and
why.

In [ ]:
# Review still costs 1 at every probability; no review now costs 10p.
threshold_if_miss_costs_10 = 1 / 10
_ = check.predicted_threshold(threshold_if_miss_costs_10)

Check your prediction with the *Missed AF* slider, which sets $C_{FN}$. The
*FP harm* slider adds any harm of a false referral beyond the review itself,
such as anxiety or a needless follow-up test, so that $C_{FP}$ = 1 + harm.

In [ ]:
expected_controls = plots.expected_cost_explorer()

In [ ]:
# Without widgets: change the two prices and rerun.
missed_af_price = 5
false_referral_harm = 0
display(plots.expected_cost_figure({"TP": 1, "FP": 1 + false_referral_harm,
                                    "FN": missed_af_price, "TN": 0}))

#### The general threshold

For any matrix, review is preferred when
$pC_{TP} + (1-p)C_{FP} \le pC_{FN} + (1-p)C_{TN}$. Write
$\text{harm} = C_{FP} - C_{TN}$, the extra loss from reviewing a recording
without AF, and $\text{benefit} = C_{FN} - C_{TP}$, the loss avoided by
reviewing a recording with AF. Rearranging gives

$$p_t = \frac{\text{harm}}{\text{harm} + \text{benefit}}.$$

With our prices this is $1/(1+4) = 0.2$. A dearer miss enlarges the benefit
and lowers $p_t$: weaker evidence now justifies a review.

Only the two differences matter. Adding the same amount to both prices in the
AF column, $C_{TP}$ and $C_{FN}$, changes no decision.

The rule "review when $p \ge p_t$" needs both harm and benefit to be positive.
Otherwise compare the expected losses directly. One action can then be at
least as good at every probability, and if both are negative the rule
reverses: review is preferred *below* a threshold.
`reference.probability_threshold` refuses such a matrix.

## 2. Six teaching recordings (about 10 minutes)

The 6 invented recordings, Teaching A to F, let you check every calculation
by hand; 3 of them contain AF. Teaching B scores high without AF and Teaching
E has AF but scores low. Neither is a contradiction: a score is uncertain
evidence. Each record is a dictionary:

| Key | Meaning |
|---|---|
| `recording` | The recording's ID. |
| `reference_label` | `A` (AF), `N` (normal rhythm) or `O` (another rhythm). |
| `af_present` | `True` when the reference label is `A`. |
| `score` | The model's score for this recording. |

In [ ]:
for record in records:
    print(f"{record['recording']}  AF: {record['af_present']!s:5}  score: {record['score']}")

### Problem 2: Predict

At a cutoff of 0.5, a recording enters review when its score is at least 0.5.
Before running anything, count the reviews and the missed AF recordings.

In [ ]:
predicted_reviews = 3  # A, B and C
predicted_misses = 1  # E
_ = check.calls_at_half(predicted_reviews, predicted_misses)

### Four rates, four denominators

At 0.5 the counts are TP=2, FP=1, FN=1 and TN=2. Each rate divides by a
different group.

| Rate | Question | Denominator | At 0.5 |
|---|---|---|---|
| Sensitivity | Of the AF recordings, what fraction is reviewed? | TP + FN | 2/3 |
| Specificity | Of those without AF, what fraction avoids review? | TN + FP | 2/3 |
| PPV | Of the reviewed recordings, what fraction has AF? | TP + FP | 2/3 |
| NPV | Of the unreviewed recordings, what fraction has no AF? | TN + FN | 2/3 |

The four agree here by coincidence. At 0.2 the counts are TP=3, FP=2, FN=0
and TN=1: sensitivity 100.0%, specificity 33.3%, PPV 60.0% and NPV 100.0%.
At 0.2, catching Teaching E means also reviewing Teaching D, which has no AF.

Sensitivity and specificity condition on the reference label. PPV and NPV
condition on the decision, so they also depend on how common AF is. The cell
below shows the same calculation with the reference functions.

In [ ]:
worked = plots.show_policy(records, .5, costs)

### From six recordings to the development cohort

The cohort uses the real CinC labels: A (AF), N (normal rhythm) and O (another
rhythm). O is AF-negative for this task but is not healthy. The 279 noisy
recordings (`~`) have no binary AF reference, so they are excluded rather than
relabelled as normal. Each label describes a whole recording, so every
decision here is made per recording. Cohort records carry two more keys,
`model` and `split`.

The 8,249 assessable recordings are split three ways.

| Split | Recordings | With AF | Used for |
|---|---:|---:|---|
| Development | 3,711 | 341 | choosing policies |
| Calibration | 1,649 | 151 | fitting the ResNet's recalibration, nothing else |
| Evaluation | 2,889 | 266 | assessing the policy you freeze in Section 8 |

In [ ]:
development = data.load_records("random_forest", "development")
print(len(development), "development recordings;",
      sum(r["af_present"] for r in development), "with reference AF.")
print(development[0])

### Problem 3: Predict

Suppose nobody is reviewed. PPV is then undefined, because there is no
reviewed group. What is the NPV on the development split? Store it as a
proportion.

In [ ]:
af = sum(r["af_present"] for r in development)
npv_if_nobody_reviewed = (len(development) - af) / len(development)
_ = check.npv_review_none(npv_if_nobody_reviewed)

#### Why NPV can look high while missing AF

Every recording is unreviewed, so NPV is the AF-free fraction of the whole
cohort: 3,370/3,711 = 90.8%. The policy misses all 341 AF recordings. NPV is
high because AF is uncommon (9.2% of this split), not because anything was
found. Report missed AF as a count beside NPV.

## 3. Price one decision (about 5 minutes)

Once the action is taken and the reference is known, the matrix gives one
realised loss. The worked example below names the matrix cell that each of
the six recordings lands in at a cutoff of 0.5. Your function in Problem 4
has the same shape.

In [ ]:
def outcome(actual_af, review):
    """Which cell of the matrix one decision lands in."""
    if review:
        if actual_af:
            return "TP"
        return "FP"
    if actual_af:
        return "FN"
    return "TN"


calls = []
for record in records:
    review = record["score"] >= .5
    calls.append({"recording": record["recording"],
                  "outcome": outcome(record["af_present"], review)})
for call in calls:
    print(call)

### Problem 4: Make

Complete `decision_cost`. It returns the price of the outcome, read from
`costs`, so the function still works when the service values outcomes
differently. The score plays no part: the decision has already been made. The
checker also tries a matrix in which no two prices are equal.

In [ ]:
def decision_cost(actual_af, review, costs):
    """The realised loss for one recording and one action."""
    if review:
        if actual_af:
            return costs["TP"]
        return costs["FP"]
    if actual_af:
        return costs["FN"]
    return costs["TN"]


_ = check.decision_cost(decision_cost)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# decision_cost = check.use_reference("decision_cost")

The explorer below prices the six recordings with **your** `decision_cost`.
At 0.5 there are 3 reviews and 1 missed AF recording, so the total loss is
3 × 1 + 1 × 5 = 8. Predict the total at 0.2 before you move the cutoff, then
change one price at a time and watch where the cheapest cutoff goes. The
explorer keeps the version of `decision_cost` it was given: if you change your
function, rerun the explorer cell.

In [ ]:
if check.ready(decision_cost=decision_cost):
    teaching_controls = plots.policy_explorer(decision_cost, records)

In [ ]:
# Without widgets: change the cutoff or the prices and rerun.
if check.ready(decision_cost=decision_cost):
    _ = plots.show_policy(records, .2, costs, decision_cost)

#### Totals for the six recordings

At 0.2 the total is 5. Reviewing all 6 costs 6, and reviewing none costs
3 × 5 = 15. The cheapest cutoff found by searching, 0.25, makes the same calls
as 0.2. A total means something only beside another policy's total on the
same recordings.

## 4. A policy and two objectives (about 15 minutes)

A policy says what to do with every recording. Keep a ledger: one row per
recording with its ID, the action and the loss, so that a colleague can audit
any total.

### Problem 5: Make

Complete `evaluate_policy`. For each record, review when its score is at or
above `threshold`, price the decision with `decision_cost`, and append a
dictionary with the keys `recording`, `review` and `loss`. Return the list.
The loop in Section 3's worked example has the same shape.

In [ ]:
def evaluate_policy(records, threshold, costs):
    """One ledger row per recording: its ID, the action and the realised loss."""
    ledger = []
    for record in records:
        review = record["score"] >= threshold
        loss = decision_cost(record["af_present"], review, costs)
        ledger.append({"recording": record["recording"], "review": review, "loss": loss})
    return ledger


if check.policy(evaluate_policy):
    plots.show_ledger(evaluate_policy(records, .5, costs))

### Youden's J

A common recipe picks the cutoff that maximises Youden's J, sensitivity plus
specificity minus 1. Before comparing it with the priced matrix, write it.

### Problem 6: Make

Complete `youden_j(counts)`, where `counts` holds `TP`, `FP`, `FN` and `TN`.

In [ ]:
def youden_j(counts):
    """Sensitivity + specificity − 1."""
    sensitivity = counts["TP"] / (counts["TP"] + counts["FN"])
    specificity = counts["TN"] / (counts["TN"] + counts["FP"])
    return sensitivity + specificity - 1


_ = check.youden_j(youden_j)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# youden_j = check.use_reference("youden_j")

### Two objectives on the development split

`reference.sweep_thresholds` tries every distinct cutoff: each recording's
score, plus 0 (review all) and 1.01 (review none). Each row it returns is one
**operating point**, a cutoff and what it does:

| Key | Meaning |
|---|---|
| `threshold` | The cutoff. |
| `TP`, `FP`, `FN`, `TN` | The four counts at that cutoff. |
| `sensitivity`, `specificity`, `ppv`, `npv`, `youden_j` | The rates and J. |
| `reviews`, `missed_af` | `TP + FP`, and `FN`. |
| `total_loss`, `mean_loss` | The loss priced with your `decision_cost`, in total and per recording. |

Several cutoffs can make identical calls, so `min` and `max` return the lowest
of any tied cutoffs.

### Problem 7: Predict

You have not yet seen how J values a miss, so this is a hypothesis. Will the
cutoff that maximises J be lower or higher than the cutoff that minimises loss
under our matrix, or the same? Store `"lower"`, `"higher"` or `"same"`. The
checker records your answer; the next cell shows the result.

In [ ]:
j_cutoff_compared_with_loss = "lower"
_ = check.j_cutoff(j_cutoff_compared_with_loss)

In [ ]:
if check.ready(decision_cost=decision_cost):
    rf_rows = reference.sweep_thresholds(development, costs, decision_cost)
    rf_min_loss = min(rf_rows, key=lambda row: row["total_loss"])
    marks = {"Minimum loss": rf_min_loss}
    j_ready = check.ready(youden_j=youden_j)
    if j_ready:
        rf_max_j = max(rf_rows, key=youden_j)
        marks["Maximum J"] = rf_max_j
    for label, row in marks.items():
        print(f"{label:12}  cutoff {row['threshold']:.3f}  reviews {row['reviews']}  "
              f"missed AF {row['missed_af']}  loss {row['total_loss']:.0f}  "
              f"J {row['youden_j']:.3f}")
    display(plots.objective_figure(rf_rows, marks))
    if j_ready:  # Your hypothesis is compared only once your J has found its cutoff.
        _ = check.reconcile_j_cutoff(j_cutoff_compared_with_loss)

J's policy reviews more recordings and misses fewer. At our prices the extra
reviews cost more than the misses they avoid, so its total loss is higher. If
your hypothesis differed, say to your partner what you assumed about J before
reading on.

#### The two policies in numbers

Maximising J chooses a cutoff of 0.102. It reviews 613 recordings, misses 52
AF recordings and costs 873. Minimising loss chooses 0.239: 410 reviews, 81
misses and a loss of 815. J's policy makes 203 extra reviews to avoid 29
misses. The reviews cost 203 and the avoided misses save 145, so it is 58
units worse at these prices.

### Why J disagrees

J weights the two error *rates* equally. There are $n_{neg}/n_{pos}$ = 9.88
recordings without AF for each AF recording, so equal weight on the rates puts
9.88 times as much weight on each miss as on each false referral. J therefore
prices a miss at 9.88 false referrals, where our matrix prices it at 4. Equal
weighting of rates is not equal pricing of errors, and J's price comes from
the case mix rather than from anyone's judgement.

#### The algebra, and what J implies

On the development split $n_{pos}$ = 341 and $n_{neg}$ = 3,370. Because
$TP = n_{pos} - FN$ and $TN = n_{neg} - FP$,

$$J = 1 - \frac{FN}{n_{pos}} - \frac{FP}{n_{neg}},$$

so maximising J is the same as minimising $\frac{n_{neg}}{n_{pos}}FN + FP$.
Our matrix gives $L = n_{pos} + FP + 4FN$: every AF recording costs at least
1 and a miss adds 4. Minimising loss is minimising $4FN + FP$.

| | One miss is priced at | Implied $p_t$ |
|---|---:|---:|
| Our matrix | 4 false referrals | 0.2 |
| Maximising J | $n_{neg}/n_{pos}$ = 9.88 false referrals | $n_{pos}/n$ = 9.2%, the prevalence |

The forest's scores are close to calibrated, so its maximum-J cutoff, 0.102,
sits near the prevalence.

Now change only the case mix, keeping each class's score distribution fixed.
Sensitivity and specificity at every cutoff stay the same, so J keeps the same
cutoff. Its implied miss price $n_{neg}/n_{pos}$ changes, though: the same
cutoff now encodes a different valuation of a miss, although nobody's view of
a missed AF recording has changed. With the prices held fixed, it is the
minimum-loss cutoff that moves.

## 5. Which model? It depends on the prices (about 10 minutes)

The ResNet ranks recordings slightly better: AUC 0.934 against 0.926 for the
forest. AUC averages over every cutoff and uses no prices. The service will
run one cutoff under one matrix, so compare each model at its own cheapest
cutoff.

The counts at each operating point do not depend on the prices; only their
loss does. One sweep per model is therefore enough. `policy_loss` reprices an
operating point with your `decision_cost`, and `cheapest` finds the best one.

**Read the output:** which model is cheaper when $C_{FN}$ = 5, and which when
$C_{FN}$ = 15?

In [ ]:
def policy_loss(row, prices):
    """Total loss of one operating point: each count times the price your function gives it."""
    return (row["TP"] * decision_cost(True, True, prices)
            + row["FP"] * decision_cost(False, True, prices)
            + row["FN"] * decision_cost(True, False, prices)
            + row["TN"] * decision_cost(False, False, prices))


def cheapest(rows, prices):
    """The operating point with the lowest total loss; the lowest cutoff wins a tie."""
    return min(rows, key=lambda row: policy_loss(row, prices))


if check.ready(decision_cost=decision_cost):
    rf_rows = reference.sweep_thresholds(development, costs, decision_cost)
    resnet_rows = reference.sweep_thresholds(data.load_records("resnet", "development"),
                                             costs, decision_cost)
    for miss in (5, 15):
        prices = {**costs, "FN": miss}
        for name, rows in [("Random forest", rf_rows), ("Raw ResNet", resnet_rows)]:
            best = cheapest(rows, prices)
            print(f"C_FN={miss:2}  {name:13}  cutoff {best['threshold']:.3f}  "
                  f"reviews {best['reviews']}  missed AF {best['missed_af']}  "
                  f"loss {policy_loss(best, prices):.0f}")

The two cheapest cutoffs differ because the score scales differ; a shared
cutoff of 0.5 would be fair to neither.

#### What the two prices show

At $C_{FN}$ = 5 the forest is cheaper: 815 against 922, at cutoffs 0.239 and
0.800. At $C_{FN}$ = 15 the ResNet is cheaper: 1,122 against 1,393. It
reviews 912 recordings and misses 14, where the forest reviews 613 and misses
52. Those 613 reviews are the policy that maximised J in Section 4. J behaves
like a matrix with a high miss price, and at this price the two objectives
choose the same cutoff.

### Problem 8: Make

Write a loop over whole-number miss prices from 2 to 30, keeping $C_{TP}$ = 1,
$C_{FP}$ = 1 and $C_{TN}$ = 0. Store in `flip` the first price at which the
raw ResNet's minimum loss is no greater than the forest's.

In [ ]:
flip = None
for miss in range(2, 31):
    prices = {"TP": 1, "FP": 1, "FN": miss, "TN": 0}
    if policy_loss(cheapest(resnet_rows, prices), prices) <= policy_loss(cheapest(rf_rows, prices), prices):
        flip = miss
        break
_ = check.flip_price(flip)

In [ ]:
if check.ready(decision_cost=decision_cost):
    # The two sweeps again, so this cell also runs on its own once decision_cost passes.
    rf_rows = reference.sweep_thresholds(development, costs, decision_cost)
    resnet_rows = reference.sweep_thresholds(data.load_records("resnet", "development"),
                                             costs, decision_cost)
    display(plots.model_choice_figure({"Random forest": rf_rows, "Raw ResNet": resnet_rows},
                                      policy_loss))

#### Where the preference flips

At $C_{FN}$ = 8 the two minimum losses are equal: 1,023 each. Below that price
the forest is cheaper and above it the ResNet is. The higher AUC does not
settle the choice; the price of a miss does. Which price is right is a
clinical and service judgement. The analysis shows which judgement the model
choice depends on, and how far apart the two models are on either side of it.

### How stable is the flip?

The flip price comes from one development split with 341 AF recordings, and
near the flip the two minimum losses differ by only a few units.

### Problem 9: Predict

On another sample of the same size from the same population, would the flip
price stay the same, or could it move? Store `"same"` or `"could_move"`.

In [ ]:
flip_on_another_sample = "could_move"
_ = check.flip_on_new_sample(flip_on_another_sample)

In [ ]:
# Resample the development recordings with replacement, keeping both models' scores
# and the label of each recording together, and find the flip price in each resample.
table = data.load_arrays("development")
flips = comparison.bootstrap_flip_prices(table["random_forest"], table["resnet"],
                                         table["af_present"], costs, range(2, 31))
for price in sorted({f for f in flips if f is not None}):
    print(f"C_FN = {price:2}  {'#' * flips.count(price)} {flips.count(price)}")
print("No flip up to C_FN = 30:", flips.count(None))

#### How far the flip moves

In 90% of 200 resamples the flip price lay between 6 and 12. One split
supports "the forest is cheaper when misses are cheap and the ResNet when they
are dear". It does not support "the flip is at exactly 8". Extension 2 builds
a paired bootstrap of this kind for a difference in net benefit.

## 6. Fix the bug (about 5 minutes)

A colleague wrote this scorer for the same matrix. It runs, returns numbers
and charges for both kinds of error. Leave this cell as it is: Problem 11
repairs a copy of it.

In [ ]:
def original_colleague_cost(actual_af, review, costs):
    """Price one decision from the four-cell matrix."""
    if actual_af and not review:
        return costs["FN"]
    if review and not actual_af:
        return costs["FP"]
    return 0

### Problem 10: Predict

Read `original_colleague_cost`. If it is used to choose a cutoff on the
development forest scores, will it pick a cutoff lower than yours, equal to
it, or higher? Store `"lower"`, `"equal"` or `"higher"`.

In [ ]:
colleague_cutoff_compared_with_yours = "lower"
_ = check.colleague_cutoff(colleague_cutoff_compared_with_yours)

In [ ]:
if check.ready(decision_cost=decision_cost):
    rf_rows = reference.sweep_thresholds(development, costs, decision_cost)  # as in Section 4
    rf_min_loss = min(rf_rows, key=lambda row: row["total_loss"])
    colleague_rows = reference.sweep_thresholds(development, costs, original_colleague_cost)
    colleague_choice = min(colleague_rows, key=lambda row: row["total_loss"])
    # Price the colleague's chosen cutoff with the full matrix.
    colleague_true = next(row for row in rf_rows
                          if row["threshold"] == colleague_choice["threshold"])
    for label, row, reported in [("Your scorer", rf_min_loss, rf_min_loss),
                                 ("Original colleague's", colleague_true, colleague_choice)]:
        print(f"{label:20}  cutoff {row['threshold']:.3f}  reviews {row['reviews']}  "
              f"missed AF {row['missed_af']}  reported loss {reported['total_loss']:.0f}  "
              f"true loss {row['total_loss']:.0f}")
    display(plots.objective_figure(rf_rows, {"Your cutoff": rf_min_loss,
                                             "Colleague's cutoff": colleague_true},
                                   "True loss of every cutoff, priced with the full matrix"))

The colleague's scorer reports a lower loss for its own choice than yours
reports, yet priced with the full matrix its choice costs more.

### Problem 11: Fix the bug

First find the smallest input that exposes the bug: call
`original_colleague_cost` and `decision_cost` on single decisions in the
scratch cell and compare. Then repair the copy below, `repaired_colleague_cost`,
so that all four outcomes are priced from `costs`. The checker uses a matrix
in which every price differs from the default.

In [ ]:
# Scratch cell: try single decisions with both functions here.

In [ ]:
def repaired_colleague_cost(actual_af, review, costs):
    """Price one decision from the four-cell matrix."""
    if actual_af and not review:
        return costs["FN"]
    if review and not actual_af:
        return costs["FP"]
    if review:
        return costs["TP"]
    return costs["TN"]


_ = check.colleague_cost(repaired_colleague_cost)

In [ ]:
# Before and after: each scorer chooses a cutoff, and the full matrix prices that choice.
if check.ready(decision_cost=decision_cost, repaired_colleague_cost=repaired_colleague_cost):
    rf_rows = reference.sweep_thresholds(development, costs, decision_cost)  # as in Section 4
    for label, scorer in [("Your scorer", decision_cost),
                          ("Original colleague's", original_colleague_cost),
                          ("Repaired colleague's", repaired_colleague_cost)]:
        choice = min(reference.sweep_thresholds(development, costs, scorer),
                     key=lambda row: row["total_loss"])
        true = next(row for row in rf_rows if row["threshold"] == choice["threshold"])
        print(f"{label:20}  cutoff {choice['threshold']:.3f}  reviews {choice['reviews']}  "
              f"reported loss {choice['total_loss']:.0f}  true loss {true['total_loss']:.0f}")

#### Why the colleague's choice moves

With the original scorer the chosen cutoff falls from 0.239 to 0.164 and
reviews rise from 410 to 498. Priced with the full matrix, that policy costs
828 rather than 815. The repaired scorer chooses 0.239 again, and its
reported loss is the true loss.

The 88 extra reviews find 15 more AF recordings and include 73 false
positives. The original scorer weighs 73 units of extra false referrals
against 75 units of avoided misses, so the change looks 2 units cheaper. It
leaves out the 15 extra correct reviews, which cost 1 each; with them the
change is 13 units dearer. Put another way, a free correct review lowers $p_t$
from 20% to 16.7%.

How much the policy moves depends on how many recordings sit near the
decision point. Here 88 recordings lie between the two chosen cutoffs. On
another sample the same mistake might move the policy less or not at all, but
it would still misstate the loss of every policy that makes a priced correct
decision. At these prices that is any policy that reviews an AF recording;
reviewing nobody is priced the same by both scorers. Here 553 is reported
against a true 828. A scorer can run and return plausible numbers while
optimising the wrong objective.

## 7. From prices to decision curves (about 10 minutes)

### Net benefit from the priced matrix

Measure each policy against reviewing nobody. Reviewing an AF recording
avoids a miss priced 5 and costs a review priced 1, a gain of 4. Reviewing a
recording without AF costs 1 more than leaving it, a loss of 1. A policy's
loss avoided is therefore $4\,TP - FP$. Divide by 4, so that the unit is the
gain from one correct review, and by the number of recordings $n$:

$$NB = \frac{TP}{n} - \frac{FP}{n}\cdot\frac{1}{4}.$$

The weight 1/4 is the odds of the threshold probability,
$0.2/(1-0.2) = 1/4$. Decision curves write the same weight through $p_t$:

$$NB(p_t) = \frac{TP}{n} - \frac{FP}{n}\cdot\frac{p_t}{1-p_t}.$$

Net benefit is measured in true-positive equivalents per recording, relative
to reviewing nobody, under the valuation that $p_t$ encodes. It is not a
count of strokes prevented.

**Worked example.** Take the six teaching recordings at $p_t$ = 0.2, reviewing
when the score is at least 0.2: TP = 3, FP = 2 and $n$ = 6.

| Term | Calculation | Value |
|---|---|---:|
| $TP/n$ | 3/6 | 0.500 |
| $FP/n \times p_t/(1-p_t)$ | 2/6 × 0.25 | 0.083 |
| $NB$ | 0.500 − 0.083 | 0.417 |
| Check: loss avoided / $4n$ | (15 − 5)/24 | 0.417 |

### Problem 12: Make

Complete `net_benefit(counts, p_t)`.

In [ ]:
def net_benefit(counts, p_t):
    """Net benefit per recording at threshold probability p_t, relative to reviewing nobody."""
    n = counts["TP"] + counts["FP"] + counts["FN"] + counts["TN"]
    return counts["TP"] / n - counts["FP"] / n * p_t / (1 - p_t)


_ = check.net_benefit(net_benefit)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# net_benefit = check.use_reference("net_benefit")

### Problem 13: Predict

At $p_t$ = 0.2 each model reviews the recordings whose stated score is at
least 0.2. Which model has the highest net benefit: `"random_forest"`,
`"resnet"` (the raw ResNet) or `"resnet_calibrated"`? Section 5 is relevant,
and so is what each model's scores mean, which the next cell shows. Record
your hypothesis; the checker does not mark it.

In [ ]:
highest_net_benefit_at_0_2 = "random_forest"
_ = check.highest_net_benefit(highest_net_benefit_at_0_2)

In [ ]:
# Each model at a stated 0.2. Your net benefit is checked against the loss avoided / 4n.
if check.ready(decision_cost=decision_cost, net_benefit=net_benefit):
    n = len(development)
    none_loss = sum(r["af_present"] for r in development) * costs["FN"]
    print(f"{'Model':28}  {'Reviews':>7}  {'Brier':>6}  {'NB':>7}  {'Loss avoided / 4n':>17}")
    for model in data.MODEL_NAMES:
        rows = data.load_records(model, "development")
        counts = reference.confusion_counts(reference.evaluate_policy(rows, .2, costs, decision_cost))
        avoided = (none_loss - reference.matrix_loss(counts, costs)) / (4 * n)
        brier = reference.calibration_summary(rows)["brier"]
        print(f"{data.MODEL_NAMES[model]:28}  {counts['TP'] + counts['FP']:7,}  {brier:6.3f}  "
              f"{net_benefit(counts, .2):7.4f}  {avoided:17.4f}")
    _ = check.reconcile_highest_net_benefit(highest_net_benefit_at_0_2)

The recalibrated ResNet is the raw ResNet's scores passed through a logistic
(Platt) map fitted on the calibration split only; Extension 1 builds it. The
map keeps the order of the scores, so both versions have the same best
achievable loss. What changes is what a stated 0.2 does. The raw scores are
too high, so at 0.2 the raw ResNet reviews far more recordings than its best
cutoff would. The Brier score, the mean squared error of the stated
probability, shows the same problem. One Brier score does not establish
calibration in every subgroup or in a new population; Extension 1 adds
reliability plots.

#### The three models at 0.2 in numbers

At 0.2 the forest has net benefit 0.059, the recalibrated ResNet 0.051 and
the raw ResNet 0.041. Reviewing everyone gives −0.135 and reviewing nobody
gives 0 by definition. Recalibration lifts the ResNet by 0.010, but the forest
remains highest at this preference. At 0.2 the raw ResNet sends 1,051
recordings for review, because its scores are too high; the recalibrated
scores send 739. An unrestricted cutoff search finds the same minimum loss for
both: 922, at cutoffs 0.800 (raw) and 0.255 (recalibrated). The Brier score
falls from 0.118 (raw) to 0.056 after recalibration; the forest's is 0.041.

### Decision curves

A decision curve repeats that policy across a range of threshold
probabilities, beside two reference strategies: review everyone and review
nobody. Read it vertically. Fix the preference first, then compare strategies
at that value. Do not look for the highest point across preferences: each
$p_t$ is a different valuation of the same outcomes.

With the other prices fixed, a preference $p_t$ corresponds to a miss price
$C_{FN} = 1/p_t$. Use the slider to find the preferences at which the
recalibrated ResNet is above the forest. Which miss prices are those, and how
do they compare with the flip price from Section 5?

In [ ]:
if check.ready(net_benefit=net_benefit):
    dca_controls = plots.decision_curve_explorer(net_benefit)

In [ ]:
# Without widgets: change the preference and rerun.
preference = .2
if check.ready(net_benefit=net_benefit):
    display(plots.decision_curves("development", preference=preference,
                                  models=["random_forest", "resnet", "resnet_calibrated"],
                                  net_benefit_function=net_benefit))

#### Where the curves cross

The recalibrated ResNet is above the forest up to a preference of about 0.12,
a miss price of about 8.3. Section 5 found the flip at a miss price of 8. The
two analyses agree because both models' stated probabilities are close to
calibrated, so acting at a stated $p_t$ is close to acting at each model's
best cutoff for that price.

## 8. Freeze and defend a policy (about 5 minutes)

Freeze the model, cutoff and prices before looking at the evaluation split.
The worked choice is the forest at its development minimum-loss cutoff; you
may choose another policy if you can defend it. Then predict the evaluation
loss. The evaluation split has 2,889 recordings and the development split
3,711, so a development total needs scaling.

### Problem 14: Predict

Set `final_model`, `final_threshold` and `final_costs`, and write down
`predicted_evaluation_loss`, the total loss you expect on the evaluation
split. Only then run the reveal cell below. Rerunning the reveal after
changing your choice is more exploration of the same data, not fresh
validation.

In [ ]:
final_model = "random_forest"
final_threshold = rf_min_loss["threshold"]
final_costs = dict(costs)
evaluation_size = data.provenance()["split_counts"]["evaluation"]
predicted_evaluation_loss = rf_min_loss["mean_loss"] * evaluation_size
frozen = check.frozen_policy(final_model, final_threshold, final_costs,
                             predicted_evaluation_loss)

In [ ]:
# Reveal: the frozen policy on the development and evaluation splits.
if not frozen:
    check.waiting("Problem 14: freeze your policy and prediction above")
elif check.ready(decision_cost=decision_cost):
    report = {split: reference.policy_summary(data.load_records(final_model, split),
                                              final_threshold, final_costs, decision_cost)
              for split in ("development", "evaluation")}
    plots.show_report(report, predicted_evaluation_loss)

Compare the evaluation loss with your prediction and with the scaled
development loss. Two things can open a gap. A new sample of recordings
varies by chance. And the cutoff was the best of many on development, so the
development loss is optimistic for that cutoff. Section 5's resampling
measured how far the flip price could move, not how far this loss can, so it
does not size this gap. A held-out split of the same cohort says nothing
about a different service.

#### The worked policy on evaluation

On evaluation the worked policy makes 339 reviews, misses 64 AF recordings
and has a loss of 659. Development's 815, scaled to the evaluation size, gives
634. The loss per recording rose from 0.220 to 0.228.

## 9. Closing discussion (about 10 minutes)

This is the module's closing session.

**First, in pairs.** State your decision in four or five sentences: the
action, your prices, your model and cutoff, and the evidence from the report.
Name one changed assumption that would make you reconsider. You may write it
in the cell below.

**Then split the room.** Half the groups take the first two questions and the
other half take the last two. Each half then reports back in two minutes.

1. **The pathway.** Who receives a flagged recording, how quickly, and what do
   they do next? Who acts on a review that finds AF, and who answers for a
   recording that was never reviewed? A review that nobody has capacity to act
   on has no benefit, whatever the matrix says.
2. **Drift and prevalence.** The prices and cutoff were chosen on one cohort.
   What changes if AF becomes commoner or rarer, or a new recording device
   shifts the scores? Which numbers in your report would you monitor?
   Extensions 1 and 4 work through recalibration and prevalence.
3. **What J assumed.** Maximising J implied a miss price set by the case mix.
   What price did it imply here, and why would that implied price change with
   prevalence when nobody's clinical judgement has?
4. **Evidence before use.** Which of these does today's analysis address, and
   which does it not?
   - a prospective evaluation on local data, with the local case mix and
     pathway;
   - monitoring of reviews, misses and calibration after go-live;
   - clinical safety management, which in England means DCB0129 for the
     manufacturer and DCB0160 for the deploying organisation;
   - a regulatory assessment of whether the software is a medical device. That
     depends on its intended purpose; the MHRA regulates devices in the UK.

**Finally, everyone.** What will you report beside AUC for your own
classifier? Extension 6 runs today's analysis on its scores.

In [ ]:
# Optional: write your decision statement here.
decision_statement = f"""
The action is to send a recording for AF review. I priced every review at
{costs['TP']} unit, including those that find AF, and a missed AF recording at
{costs['FN']}, which makes review worthwhile above an AF probability of
{reference.probability_threshold(costs):.0%}. At those prices the random forest at its
development minimum-loss cutoff was cheaper than the raw ResNet at its own best
cutoff, despite the ResNet's higher AUC, so I froze that policy before opening
the evaluation split. The report gives the reviews, the missed AF recordings and
the total loss under that matrix. If a missed AF recording were priced at about
{flip} units or more, I would reconsider the ResNet, although resampling showed
that this flip price is uncertain. These are simulated scores, so the report
illustrates the reasoning and does not estimate the value of a real service.
"""
_ = check.reflection(decision_statement)

## Continue with the extensions

The extensions are in the `extension/` folder. Each uses the same matrix and
the reference versions of this notebook's functions, so none depends on your
answers here. Choose one.

| | Extension | The question it answers | Time |
|---|---|---|---|
| 1 | Calibration and Platt scaling | How do we make raw scores into probabilities a threshold can act on? | about 40 minutes |
| 2 | Comparing models: reclassification, NRI, IDI and net-benefit differences | Which recordings change decision between two models, and does the change help? | about 45 minutes |
| 3 | Class weighting versus moving the threshold | Does reweighting the training loss do anything a different cutoff cannot? | about 30 minutes |
| 4 | Capacity, prevalence and daily queues | What happens when AF is rarer, or the review team can only see so many recordings a day? | about 35 minutes |
| 5 | Window-level calls and unreadable recordings | How should window scores become one recording-level call, and what should happen to a recording nobody can read? | about 40 minutes |
| 6 | Your own classifier's scores | What does today's analysis say about the classifier you trained this week? | about 20 minutes |

If Section 7 was hard, take Extension 1. To compare models more carefully,
take Extension 2. If deployment interests you, take Extension 4. If you have
scores from your own classifier, take Extension 6.